# Batch-size test: which regime is the training step in?

Times the **baseline training step** (one GPU, default AdamW, `.item()` every batch; the inner loop of `train()` in `src/smi/models/model.py`) at batch sizes 32, 64, 128 and 256, on real train batches (k = 0).

`FP16` in the setup cell switches the same step to fp16 autocast + loss scaling. Everything else is unchanged, so fp32 vs fp16 differs by that one thing.

**How to read it** (concepts/performance-analysis-and-optimization/02):
- time per step grows **proportionally** with the batch → the step is compute- or memory-bound;
- time per step grows **much less** than the batch → fixed per-step costs dominate (launch overhead, plus the optimizer and clipping, which do not depend on the batch).

Two timing modes for every batch size:
- **total:** no extra synchronisation, the real time per step;
- **breakdown:** `torch.cuda.synchronize()` between phases, the time of each phase. The syncs remove the CPU/GPU overlap, so this total is a bit higher.

An exploration notebook (PROTOCOL §8): its numbers inform decisions, they are not reportable results.

Settings: **Accelerator GPU T4** (one GPU is used), **Internet on**, the private `smi-v4-meetings` dataset attached.

## 1. GPU, code, packages, data

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /kaggle/working/repo
!git clone -b C/baselines-design https://github.com/ahmedwaleedaref/Meeting-Intelligent-System-.git /kaggle/working/repo
!cd /kaggle/working/repo && git log --oneline -1
!pip install -q transformers==5.20.1 sentencepiece==0.2.2

In [ ]:
import glob, os, sys
sys.path.insert(0, "/kaggle/working/repo/src")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

hits = glob.glob("/kaggle/input/**/source.json", recursive=True)
assert len(hits) == 1, f"expected exactly one source.json under /kaggle/input, found {hits}"
DATA_ROOT = os.path.dirname(hits[0])
MODEL = "microsoft/deberta-v3-base"
FP16 = True  # False = the fp32 baseline step ; True = the same step under fp16 autocast + loss scaling
FUSED = True  # False = the default AdamW (foreach) ; True = fused AdamW (the whole update in a few large kernels)
print("DATA_ROOT =", DATA_ROOT, "| FP16 =", FP16, "| FUSED =", FUSED)

## 2. The step being measured
Exactly the baseline loop's inner step: forward → cross-entropy → backward → `loss.item()` → clip → AdamW step → scheduler step → `zero_grad`.
One model and one optimizer are reused for all batch sizes: the weights change while we measure, which does not affect speed.

In [ ]:
import time
from pathlib import Path

import torch
from torch.profiler import record_function
from transformers import get_scheduler

from smi.inputs.collator import SegmentCollator
from smi.inputs.dataset import SegmentDataset
from smi.inputs.sampler import LengthBucketBatchSampler
from smi.models.model import build_optimizer, load_model_and_tokenizer

device = torch.device("cuda")
torch.manual_seed(42)
model, tokenizer = load_model_and_tokenizer(MODEL)
model.to(device)
model.train()
optimizer = build_optimizer(model, 2.0e-5, 0.01)  # the default AdamW (foreach) , as in the baseline
if FUSED:
    # the same two parameter groups (weight decay on 2-D parameters only) , fused implementation
    optimizer = torch.optim.AdamW(
        [{"params": g["params"], "weight_decay": g["weight_decay"]} for g in optimizer.param_groups],
        lr=2.0e-5,
        fused=True,
    )
scheduler = get_scheduler("linear", optimizer, num_warmup_steps=100, num_training_steps=100_000)
loss_function = torch.nn.CrossEntropyLoss()
# fp16 : multiplies the loss before backward so small fp16 gradients do not round to 0 ;
# when FP16 is False the scaler does nothing and the step is exactly the fp32 baseline
scaler = torch.amp.GradScaler("cuda", enabled=FP16)
skipped = {"steps": 0}
NO_ITEM = False  # set by the run cell : True = the loss is summed on the GPU , no .item() per batch
loss_sum = {"cpu": 0.0, "gpu": torch.zeros((), device=device)}

dataset = SegmentDataset(Path(DATA_ROOT) / "train", 0, 0)
collator = SegmentCollator(tokenizer, 512, "\n", markers=("[T] ", " [/T]"))
print(f"{len(dataset)} train segments | model on {device} | weights {model.dtype} | FP16 autocast {FP16} | "
      f"AdamW fused {optimizer.defaults.get('fused')}")


def get_batches(batch_size, count):
    """the first `count` full batches of the training sampler (same seed as the runs)."""
    sampler = LengthBucketBatchSampler(dataset, batch_size, 50, True, seed=42)
    batches = []
    for indices in sampler:
        if len(indices) == batch_size:
            batches.append(indices)
        if len(batches) == count:
            break
    return batches


def one_step(indices, breakdown, phases):
    """one baseline training step ; adds each phase's seconds to `phases` ; returns the padded length."""
    t0 = time.perf_counter()
    batch = collator([dataset[i] for i in indices])
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)
    if breakdown:
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    # matrix multiplications in fp16 on Tensor Cores ; weights stay fp32
    # record_function only names the phase in a profiler trace ; it does nothing when no profiler runs
    with record_function("forward"):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=FP16):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        loss = loss_function(logits.float(), labels)
    if breakdown:
        torch.cuda.synchronize()
    t2 = time.perf_counter()

    with record_function("backward"):
        scaler.scale(loss).backward()  # = loss.backward() when FP16 is False
        if NO_ITEM:
            loss_sum["gpu"] = loss_sum["gpu"] + loss.detach()  # stays on the GPU : no sync
        else:
            loss_sum["cpu"] += loss.item()  # the baseline's GPU -> CPU sync on every batch
    if breakdown:
        torch.cuda.synchronize()  # without it , unfinished backward work would be counted in the clip phase
    t3 = time.perf_counter()

    with record_function("clip"):
        scaler.unscale_(optimizer)  # divide the gradients by the scale again (a pass over all gradients)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    if breakdown:
        torch.cuda.synchronize()
    t4 = time.perf_counter()

    # get_scale() is scale.item() : a sync , so the skipped-step counter runs in breakdown mode only
    if breakdown and FP16:
        scale_before = scaler.get_scale()
    with record_function("optimizer"):
        scaler.step(optimizer)  # = optimizer.step() when FP16 is False ; skipped if a gradient overflowed
        scaler.update()
    if breakdown and FP16 and scaler.get_scale() < scale_before:
        skipped["steps"] += 1  # the scale is lowered exactly when a step was skipped
    scheduler.step()
    optimizer.zero_grad()
    if breakdown:
        torch.cuda.synchronize()
    t5 = time.perf_counter()

    phases["data"] += t1 - t0
    phases["forward"] += t2 - t1
    phases["backward"] += t3 - t2
    phases["clip"] += t4 - t3
    phases["optimizer"] += t5 - t4
    return input_ids.shape[1]


def measure(batch_size, warmup=10, timed=60, breakdown_steps=30):
    batches = get_batches(batch_size, warmup + timed + breakdown_steps)
    torch.cuda.reset_peak_memory_stats()
    phases = {"data": 0.0, "forward": 0.0, "backward": 0.0, "clip": 0.0, "optimizer": 0.0}

    for indices in batches[:warmup]:  # not timed : first kernels , memory allocator , caches
        one_step(indices, False, phases)

    # total mode : no extra syncs
    lengths = []
    torch.cuda.synchronize()
    start = time.perf_counter()
    for indices in batches[warmup:warmup + timed]:
        lengths.append(one_step(indices, False, phases))
    torch.cuda.synchronize()
    seconds_per_step = (time.perf_counter() - start) / timed

    # breakdown mode : a sync between phases
    for name in phases:
        phases[name] = 0.0
    for indices in batches[warmup + timed:]:
        one_step(indices, True, phases)

    padded_length = sum(lengths) / len(lengths)
    return {
        "batch": batch_size,
        "ms_per_step": 1000 * seconds_per_step,
        "padded_length": padded_length,
        "tokens_per_step": batch_size * padded_length,
        "min_per_epoch": (78903 / batch_size) * seconds_per_step / 60,
        "peak_gb": torch.cuda.max_memory_allocated() / 1e9,
        "fp16": FP16,
        "no_item": NO_ITEM,
        "phases_ms": {name: 1000 * value / breakdown_steps for name, value in phases.items()},
    }

## 3. Run

In [ ]:
# warm-up pass , discarded : the first measurement also pays one-time costs
# (kernel selection for new tensor shapes , memory allocator growth)
for batch_size in (32, 64):
    measure(batch_size, warmup=10, timed=5, breakdown_steps=1)
print("warm-up done", flush=True)

# the two variants in the same session (same GPU , same batches) : they differ only by NO_ITEM
results = []
for NO_ITEM in (False, True):
    for batch_size in (32, 64):
        try:
            r = measure(batch_size)
            results.append(r)
            print(f"no_item {NO_ITEM!s:5} | batch {batch_size:>3}: {r['ms_per_step']:7.1f} ms/step | "
                  f"{r['min_per_epoch']:5.1f} min/epoch | peak {r['peak_gb']:.1f} GB", flush=True)
        except torch.OutOfMemoryError:
            print(f"no_item {NO_ITEM!s:5} | batch {batch_size:>3}: out of memory", flush=True)
            torch.cuda.empty_cache()


## 4. Results

In [ ]:
print("precision:", "fp16 autocast" if FP16 else "fp32", "| AdamW fused:", FUSED,
      "| fp16 skipped steps (breakdown mode):", skipped["steps"])
print(f"{'no_item':>7} | {'batch':>5} | {'ms/step':>8} | {'ms/1k tokens':>12} | {'min/epoch':>9} | {'peak GB':>7}")
for r in results:
    print(f"{r['no_item']!s:>7} | {r['batch']:>5} | {r['ms_per_step']:>8.1f} | "
          f"{1000 * r['ms_per_step'] / r['tokens_per_step']:>12.2f} | {r['min_per_epoch']:>9.1f} | {r['peak_gb']:>7.1f}")

names = list(results[0]["phases_ms"])
print(f"\n{'no_item':>7} | {'batch':>5} | " + " | ".join(f"{name:>9}" for name in names) + " | breakdown total (ms)")
for r in results:
    p = r["phases_ms"]
    print(f"{r['no_item']!s:>7} | {r['batch']:>5} | " + " | ".join(f"{p[name]:>9.1f}" for name in names)
          + f" | {sum(p.values()):>9.1f}")


**Reading the tables**
- `x time` vs `x tokens`: if time grows much less than the tokens, fixed costs dominate.
- `clip` and `optimizer` should stay roughly constant across batch sizes: they touch all 184M parameters whatever the batch.
- `forward` and `backward` are the parts that grow with the batch. How fast they grow says whether the model itself is overhead-bound or compute-bound at these sizes.
- `data` is the CPU's batch preparation (tokenizing + copying to the GPU).

## 5. Is the GPU busy? (`torch.profiler`)

`torch.profiler.profile(activities=[CPU, CUDA])` records every operation on the CPU and every activity on the GPU (kernels, memsets, copies), each with its start and end time.
- **GPU busy time** = the union of all GPU activity intervals. The gaps between them are time the GPU sat idle, waiting for the CPU to launch the next kernel.
- **Utilisation** = busy time ÷ the real time per step. The real time is measured on the same batches **without** the profiler, because the profiler slows the CPU down.
- `record_function("forward")` etc. in `one_step` name the phases, so the trace shows them.
- The trace of batch 32 is saved to `/kaggle/working/trace_batch32.json`. Download it and open it in **https://ui.perfetto.dev** (drag the file in): the top rows are the CPU (Python launching), the lower rows the GPU (kernels running).

Setting: the current best step (fp16 + fused AdamW + no `.item()`).

In [ ]:
from torch.profiler import ProfilerActivity, profile


def gpu_busy(batch_size, warmup=10, timed=20, profiled=10, trace_path=None):
    global NO_ITEM
    NO_ITEM = True
    batches = get_batches(batch_size, warmup + timed + profiled)
    phases = {"data": 0.0, "forward": 0.0, "backward": 0.0, "clip": 0.0, "optimizer": 0.0}

    for indices in batches[:warmup]:
        one_step(indices, False, phases)

    # the real time per step , without the profiler
    torch.cuda.synchronize()
    start = time.perf_counter()
    for indices in batches[warmup:warmup + timed]:
        one_step(indices, False, phases)
    torch.cuda.synchronize()
    real_ms = 1000 * (time.perf_counter() - start) / timed

    # the same kind of steps under the profiler
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
        for indices in batches[warmup + timed:]:
            one_step(indices, False, phases)
        torch.cuda.synchronize()

    # GPU-side events : real work (kernels , memsets , copies) and the phase annotations of record_function.
    # an annotation spans its whole phase INCLUDING idle gaps , so it must not be counted as work.
    work = []
    annotations = []
    for event in prof.events():
        if event.device_type.name != "CUDA":
            continue
        interval = (event.time_range.start, event.time_range.end)  # microseconds
        if event.is_user_annotation:
            annotations.append((event.name, interval))
        else:
            work.append(interval)
    work.sort()

    # busy time = the union of the work intervals (merged where they overlap)
    busy_us = 0.0
    current_start, current_end = work[0]
    for interval_start, interval_end in work[1:]:
        if interval_start > current_end:
            busy_us += current_end - current_start
            current_start, current_end = interval_start, interval_end
        else:
            current_end = max(current_end, interval_end)
    busy_us += current_end - current_start

    # GPU work per phase : the work intervals that start inside that phase's annotation span
    phase_gpu_ms = {"forward": 0.0, "backward": 0.0, "clip": 0.0, "optimizer": 0.0}
    for name, (span_start, span_end) in annotations:
        if name in phase_gpu_ms:
            for interval_start, interval_end in work:
                if span_start <= interval_start < span_end:
                    phase_gpu_ms[name] += (interval_end - interval_start) / 1000 / profiled

    if trace_path is not None:
        prof.export_chrome_trace(trace_path)

    busy_ms = busy_us / 1000 / profiled
    return {
        "batch": batch_size,
        "real_ms_per_step": real_ms,
        "gpu_busy_ms_per_step": busy_ms,
        "gpu_idle_ms_per_step": real_ms - busy_ms,
        "utilisation": busy_ms / real_ms,
        "gpu_activities_per_step": len(work) / profiled,
        "phase_gpu_ms": phase_gpu_ms,
    }


busy = []
for batch_size in (32, 64):
    trace = "/kaggle/working/trace_batch32.json" if batch_size == 32 else None
    r = gpu_busy(batch_size, trace_path=trace)
    busy.append(r)
    print(f"batch {batch_size:>3}: real {r['real_ms_per_step']:6.1f} ms/step | GPU busy {r['gpu_busy_ms_per_step']:6.1f} ms | "
          f"idle {r['gpu_idle_ms_per_step']:6.1f} ms | utilisation {100 * r['utilisation']:5.1f}% | "
          f"{r['gpu_activities_per_step']:.0f} GPU activities per step", flush=True)
    print("           GPU time per phase (ms):", {k: round(v, 1) for k, v in r["phase_gpu_ms"].items()}, flush=True)

**Reading it**
- **Utilisation close to 100%:** the GPU is working nearly all the time, so the step is GPU-bound. On one GPU, only less GPU work (a smaller model, fewer FLOPs) makes it faster.
- **Utilisation well below 100%:** the GPU waits for the CPU part of every step. The levers are fewer kernel launches (`torch.compile`, CUDA graphs) or more work per launch (larger batches).
- **GPU time per phase vs the breakdown table:** where the two differ a lot, that phase's time is mostly CPU-side (launching), not GPU work.